# What's in the lakehouse

A look at every Delta table that exists right now: what's there, how big it is, when it was last written, and what's inside bronze.

Tables are **discovered**, not listed by hand — any folder under the lakehouse root with a `_delta_log` is a Delta table — so this notebook keeps working as Phase 2 adds silver and Phase 5 adds gold.

Read-only: nothing here writes to a table.

In [ ]:
import json
import os

import pandas as pd
from pyspark.sql import functions as F

from ingestion import paths
from ingestion.spark import get_spark

spark = get_spark("lakehouse-tables")
spark.sparkContext.setLogLevel("ERROR")
pd.set_option("display.max_colwidth", 80)

## 1. Catalog: every table, one row each

`DESCRIBE DETAIL` reads Delta's metadata, not the data — so it's instant even on a big table. `numFiles` is worth watching: thousands of tiny files is the "small files problem" that slows Spark down.

In [ ]:
def find_delta_tables(root):
    for folder, subfolders, _ in os.walk(root):
        if "_delta_log" in subfolders:
            subfolders.clear()  # don't descend into a table's own folders
            yield folder

catalog = []
for path in sorted(find_delta_tables(paths.LAKEHOUSE_ROOT)):
    detail = spark.sql(f"DESCRIBE DETAIL delta.`{path}`").first()
    catalog.append({
        "table": os.path.relpath(path, paths.LAKEHOUSE_ROOT),
        "rows": spark.read.format("delta").load(path).count(),
        "files": detail.numFiles,
        "size_kb": round(detail.sizeInBytes / 1024, 1),
        "last_write_utc": detail.lastModified,
        "path": path,
    })
catalog = pd.DataFrame(catalog)
catalog.drop(columns="path")

## 2. Schemas

The declared schema of each table. `nullable = false` is a constraint Delta enforces on every write.

In [ ]:
for row in catalog.itertuples():
    print(f"── {row.table}")
    spark.read.format("delta").load(row.path).printSchema()

## 3. Commit history

Every write is a numbered commit. Useful for "what ran, when, and what did it do?" — and the basis of time travel (`versionAsOf`).

In [ ]:
for row in catalog.itertuples():
    print(f"── {row.table}")
    spark.sql(f"DESCRIBE HISTORY delta.`{row.path}`").select(
        "version", "timestamp", "operation", "operationMetrics.numOutputRows"
    ).show(10, truncate=False)

## 4. Bronze: what was loaded, by whom, for which dates

One row per vendor response. Grouping by `run_id` shows each load; a date fetched twice shows up as two runs — bronze keeps both deliveries on purpose.

In [ ]:
bronze = {source: spark.read.format("delta").load(path) for source, path in paths.BRONZE_BARS.items()
          if os.path.isdir(os.path.join(path, "_delta_log"))}

for source, df in bronze.items():
    print(f"── {source}")
    df.groupBy("run_id", "request_start", "request_end").agg(
        F.count("*").alias("responses"),
        F.countDistinct("security_id").alias("securities"),
        F.sum("record_count").alias("bars"),
        F.min("ingested_at").alias("ingested_at"),
    ).orderBy("ingested_at").show(truncate=False)

Responses that came back **empty** (`record_count = 0`). On a trading day this is a missing security; silver's completeness check (step 3) will turn it into a failure.

In [ ]:
for source, df in bronze.items():
    empty = df.where("record_count = 0").select("security_id", "vendor_symbol", "request_start", "run_id")
    print(f"{source}: {empty.count()} empty responses")
    empty.show(10, truncate=False)

## 5. Inside one payload

The raw text each vendor sent for Berkshire Class B, plus the request that produced it. Note the two spellings (`BRK.B` / `BRK-B`) — both rows carry the same `security_id`.

In [ ]:
BRK_B = "BBG000DWG505"
for source, df in bronze.items():
    r = df.where(F.col("security_id") == BRK_B).orderBy(F.desc("ingested_at")).first()
    if r is None:
        continue
    print(f"── {source}  (as '{r.vendor_symbol}', run {r.run_id})")
    print("request:", r.request_params)
    print("payload:", r.payload[:400], "\n")

## 6. Silver: what's there

Bronze holds what vendors *said*; silver holds what it *means* — one typed row per (`security_id`, `trade_date`, `source`), prices as exact decimals. From here on nothing in this notebook parses JSON: silver already did, with tests.

In [ ]:
bars = spark.read.format("delta").load(paths.SILVER_DAILY_BARS)
master = spark.read.format("delta").load(paths.SECURITY_MASTER).where("is_current").select("security_id", "exchange_ticker")

bars.groupBy("source", "price_basis").agg(
    F.count("*").alias("bars"),
    F.countDistinct("security_id").alias("securities"),
    F.countDistinct("trade_date").alias("sessions"),
    F.min("trade_date").alias("first"),
    F.max("trade_date").alias("last"),
).show(truncate=False)

Bars per year. A full year is ~252 sessions × 80 securities ≈ 20,000 per source; the first and last years are partial because the window is 5 years to the day.

In [ ]:
bars.groupBy(F.year("trade_date").alias("year")).pivot("source").count().orderBy("year").show()

`revision_count` counts vendor corrections: a newer delivery with different values. All zeros means no vendor has changed a bar after first delivering it — yet.

In [ ]:
bars.groupBy("source", "revision_count").count().orderBy("source", "revision_count").show()

## 7. One bar, two vendors

The last five sessions of Berkshire Class B. Closes agree to the cent; volumes don't quite (Alpaca's runs ~0.5–0.8% higher across the universe). `vwap` and `trade_count` exist only for Alpaca.

In [ ]:
BRK_B = "BBG000DWG505"
bars.where(F.col("security_id") == BRK_B).orderBy(F.desc("trade_date"), "source").select(
    "trade_date", "source", "price_basis", "open", "close", "volume", "vwap", "trade_count"
).show(10)

## 8. Corporate actions

Dividends and splits come from Yahoo; reorganizations come from SEC filings (`source = sec_edgar`), each traceable to its Form 8-K12B through `source_run_id` (the accession number).

In [ ]:
actions = spark.read.format("delta").load(paths.SILVER_CORPORATE_ACTIONS).join(master, "security_id")
actions.groupBy("action_type", "source").count().orderBy("action_type").show()

In [ ]:
print("Splits as Yahoo reports them -- whole ratios are true splits, fractional ones are spin-offs:")
actions.where("action_type = 'split'").select("exchange_ticker", "ex_date", "value").orderBy("ex_date").show(30)

print("Reorganizations, with their SEC evidence:")
actions.where("action_type = 'reorg'").select(
    "exchange_ticker", "ex_date", "value", "source_run_id", "first_seen_at"
).orderBy("ex_date").show(truncate=False)

## 9. What the backfill found

Alpaca's prices are raw; Yahoo's are adjusted by Yahoo's own rules. Dividing one close by the other on the same day shows exactly which adjustments Yahoo applied: the ratio is 1.00 when nothing happened after that date, and the split ratio before a split.

In [ ]:
a = bars.where("source = 'alpaca'").select("security_id", "trade_date", F.col("close").alias("alpaca_close"))
y = bars.where("source = 'yahoo'").select("security_id", "trade_date", F.col("close").alias("yahoo_close"))
ratio = (a.join(y, ["security_id", "trade_date"])
          .withColumn("ratio", F.round(F.col("alpaca_close") / F.col("yahoo_close"), 2))
          .join(master, "security_id"))

print("NVIDIA around its 10-for-1 split on 2024-06-10:")
ratio.where("exchange_ticker = 'NVDA' AND trade_date BETWEEN '2024-06-05' AND '2024-06-11'").select(
    "trade_date", "alpaca_close", "yahoo_close", "ratio"
).orderBy("trade_date").show()

In [ ]:
print("The ratio across every security-day:")
ratio.groupBy("ratio").count().orderBy(F.desc("count")).show(12)

How to read it:

- **1.00** — nothing adjusted after that day. Most rows.
- **10, 25, 5, 3, 20** — true splits (NVDA/AVGO/NFLX 10, BKNG 25, NOW 5, TSLA/WMT 3, AMZN/GOOGL/GOOG 20).
- **1.06, 1.13, 1.25, 1.61, …** — spin-offs, which Yahoo records as fractional "splits" and adjusts for. GE's two (1.281 × 1.253) compound to 1.61. This is why Yahoo's `price_basis` is `vendor_adjusted`, and why gold has to classify Yahoo's "splits" before using them.
- **1.01** — HON only, on every day before 2025-10-30: Yahoo's two HON adjustments compound, 1.061 × 0.9535 = 1.0117. Not vendor disagreement. The 0.9535 itself — a ratio *below* 1, which raises past prices — is still unexplained; the cell below shows where 1.01 occurs.

In [ ]:
ratio.where("ratio = 1.01").groupBy("exchange_ticker").agg(
    F.count("*").alias("days"), F.min("trade_date").alias("from"), F.max("trade_date").alias("to")
).orderBy(F.desc("days")).show(10)

In [ ]:
spark.stop()